In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=2)
search_results = search.invoke("what is the weather in SF")
print(search_results)
# If we want, we can create other tools.
# Once we have all the tools we want, we can put them in a list that we will reference later.
tools = [search]

[{'url': 'https://world-weather.info/forecast/usa/san_francisco/december-2024/', 'content': 'Extended weather forecast in San Francisco. Hourly Week 10 days 14 days 30 days Year. Detailed ⚡ San Francisco Weather Forecast for December 2024 - day/night 🌡️ temperatures, precipitations - World-Weather.info.'}, {'url': 'https://www.accuweather.com/en/us/san-francisco/94103/december-weather/347629', 'content': 'World North America United States California San Francisco. Get the monthly weather forecast for San Francisco, CA, including daily high/low, historical averages, to help you plan ahead.'}]


In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI()

In [6]:
from langchain_core.messages import HumanMessage

response = model.invoke(
    [
        HumanMessage(
            content="hi!",
        )
    ]
)
response.content

'Hello! How can I assist you today?'

In [7]:
model_with_tools = model.bind_tools(tools)

In [8]:
response = model_with_tools.invoke(
    [
        HumanMessage(
            content="Hi!",
        )
    ]
)

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: Hello! How can I assist you today?
ToolCalls: []


In [9]:
response = model_with_tools.invoke(
    [
        HumanMessage(
            content="What's the weather in SF?",
        )
    ]
)

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_Wax3MqMlwspKZBSxQJkBCbqQ', 'type': 'tool_call'}]


In [10]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(
    model,
    tools,
)

In [11]:
response = agent_executor.invoke(
    {
        "messages": [
            HumanMessage(
                content="hi!",
            )
        ]
    }
)

response["messages"]

[HumanMessage(content='hi!', id='1b54c371-9f11-44d2-81d0-717b30fe74e4'),
 AIMessage(content='Hello! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 83, 'total_tokens': 93}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-3f4e495c-c02c-495e-ac55-d927c6460877-0', usage_metadata={'input_tokens': 83, 'output_tokens': 10, 'total_tokens': 93})]

In [12]:
response = agent_executor.invoke(
    {
        "messages": [
            HumanMessage(
                content="whats the weather in sf?",
            )
        ]
    }
)
response["messages"]

[HumanMessage(content='whats the weather in sf?', id='3659d60a-2b16-405d-8ce3-5b74c6c151b7'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_hhPFqY4zBk62uni10xdegdZB', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 88, 'total_tokens': 109}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-7d2dda8d-379f-4354-878a-623ff1190494-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_hhPFqY4zBk62uni10xdegdZB', 'type': 'tool_call'}], usage_metadata={'input_tokens': 88, 'output_tokens': 21, 'total_tokens': 109}),
 ToolMessage(content='[{"url": "https://www.timeanddate.com/weather/usa/san-francisco/historic", "content": "San Francisco Weather History for the Previous 24 Hours Show weath

In [13]:
for chunk in agent_executor.stream(
    {
        "messages": [
            HumanMessage(
                content="whats the weather in sf?",
            )
        ]
    }
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_aFCHq1Jn6AIbtkqLRTvCkkSR', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 88, 'total_tokens': 109}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d8551c32-330a-4404-87e4-e409fc1c0710-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_aFCHq1Jn6AIbtkqLRTvCkkSR', 'type': 'tool_call'}], usage_metadata={'input_tokens': 88, 'output_tokens': 21, 'total_tokens': 109})]}}
----
{'tools': {'messages': [ToolMessage(content='[{"url": "https://www.weatherapi.com/", "content": "{\'location\': {\'name\': \'San Francisco\', \'region\': \'California\', \'country\': \'United States of America\', \'lat\': 37.78, \'lon\': -

In [14]:
async for event in agent_executor.astream_events(
    {
        "messages": [
            HumanMessage(
                content="whats the weather in sf?",
            )
        ]
    },
    version="v1",
):
    kind = event["event"]
    if kind == "on_chain_start":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print(
                f"Starting agent: {event['name']} with input: {event['data'].get('input')}"
            )
    elif kind == "on_chain_end":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print()
            print("--")
            print(
                f"Done agent: {event['name']} with output: {event['data'].get('output')['output']}"
            )
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(
                content,
                end="|",
            )
    elif kind == "on_tool_start":
        print("--")
        print(
            f"Starting tool: {event['name']} with inputs: {event['data'].get('input')}"
        )
    elif kind == "on_tool_end":
        print(f"Done tool: {event['name']}")
        print(f"Tool output was: {event['data'].get('output')}")
        print("--")

/Users/samsepiol/Documents/Langchain_Use_cases/venv/lib/python3.12/site-packages/langchain_core/_api/beta_decorator.py:87: LangChainBetaWarning: This API is in beta and may change in the future.
  warn_beta(


--
Starting tool: tavily_search_results_json with inputs: {'query': 'weather in San Francisco'}
Done tool: tavily_search_results_json
Tool output was: [{'url': 'https://forecast.weather.gov/zipcity.php?inputstring=San+Francisco,CA', 'content': 'Detailed Forecast. Mostly clear, with a low around 57. West southwest wind 5 to 10 mph becoming light after midnight. Sunny, with a high near 80. Light southwest wind becoming west southwest 5 to 10 mph in the afternoon. Mostly clear, with a low around 58. Southwest wind 5 to 8 mph. Sunny, with a high near 76.'}, {'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.78, 'lon': -122.42, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1720772400, 'localtime': '2024-07-12 1:20'}, 'current': {'last_updated_epoch': 1720772100, 'last_updated': '2024-07-12 01:15', 'temp_c': 17.8, 'temp_f': 64.0, 'is_day': 0, 'condition': {'text': 'Clear', 'ic

In [15]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [16]:
agent_executor = create_react_agent(
    model,
    tools,
    checkpointer=memory,
)

config = {
    "configurable": {
        "thread_id": "abc123",
    }
}

In [17]:
for chunk in agent_executor.stream(
    {
        "messages": [
            HumanMessage(
                content="hi im bob!",
            )
        ]
    },
    config,
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='Hello Bob! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 85, 'total_tokens': 96}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-54db4351-20cc-4749-bb0c-b31fde31dc75-0', usage_metadata={'input_tokens': 85, 'output_tokens': 11, 'total_tokens': 96})]}}
----


In [18]:
for chunk in agent_executor.stream(
    {
        "messages": [
            HumanMessage(
                content="whats my name?",
            )
        ]
    },
    config,
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='Your name is Bob! How can I help you, Bob?', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 108, 'total_tokens': 122}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-f82f0bea-ebc5-434c-9aee-f1bad8453226-0', usage_metadata={'input_tokens': 108, 'output_tokens': 14, 'total_tokens': 122})]}}
----


In [19]:
config = {
    "configurable": {
        "thread_id": "xyz123",
    }
}
for chunk in agent_executor.stream(
    {
        "messages": [
            HumanMessage(
                content="whats my name?",
            )
        ]
    },
    config,
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content="I can't access personal information like your name. How can I assist you today?", response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 86, 'total_tokens': 104}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-8e9221a2-ff63-456e-a5cb-a03d6c39b558-0', usage_metadata={'input_tokens': 86, 'output_tokens': 18, 'total_tokens': 104})]}}
----
